# 2 · Training & hyperparameter analysis

Three tabular approaches on the same split:

| Model | Idea | Hyperparameter study |
|-------|------|----------------------|
| **XGBoost** | manual gradient-boosting baseline | fixed config |
| **TabPFN-3** | zero-shot pre-trained transformer | `n_estimators` sweep |
| **AutoGluon** | AutoML stack-ensemble | 2x2 grid (`num_bag_folds` x `num_stack_levels`) |

Run notebook **1** first so `results/splits.pkl` exists.

> **TabPFN-3 token:** it needs a one-time Prior Labs license token. Set
> `TABPFN_TOKEN` as an environment variable, or as a Colab secret named
> `TABPFN_TOKEN` with *Notebook access* enabled. `models.make_tabpfn` loads it
> automatically.

In [ ]:
# Make the credit_rating package importable whether or not it's pip-installed.
try:
    import credit_rating  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path.cwd().parent))

from credit_rating import config, data, models, evaluate, experiments, plots
print("Package loaded. Device:", config.get_device())

In [ ]:
X_train, X_test, y_train, y_test = data.load_splits()
print("Loaded splits:", X_train.shape, X_test.shape)

## 2.1 · XGBoost baseline

In [ ]:
xgb_model, xgb_res = experiments.run_xgboost_baseline(X_train, X_test, y_train, y_test)
print({k: round(v, 4) for k, v in xgb_res.items() if k in ("accuracy","f1_macro","f1_weighted")})
evaluate.print_classification_report(y_test, xgb_res["y_pred"], "XGBoost - per-class report")

In [ ]:
evaluate.plot_confusion_matrix(y_test, xgb_res["y_pred"], "XGBoost", cmap="Blues",
                               save_path=config.FIGURES_DIR / "xgb_confusion_matrix.png")
evaluate.plot_feature_importance(xgb_res["feature_importance"], "XGBoost - Top 15 Feature Importances",
                                 color="steelblue", kind="bar",
                                 save_path=config.FIGURES_DIR / "xgb_feature_importance.png")

## 2.2 · TabPFN-3 — `n_estimators` sweep

Each estimator views a different random feature subset; predictions are averaged.
With only 16 features we expect gains to plateau quickly — the sweep shows where.

In [ ]:
sweep_df = experiments.tabpfn_n_estimators_sweep(X_train, X_test, y_train, y_test,
                                                n_values=(1, 2, 4, 8, 16))
sweep_df

In [ ]:
plots.plot_estimator_sweep(sweep_df, save_path=config.FIGURES_DIR / "tabpfn3_estimators_plot.png")
plots.plot_marginal_gain(sweep_df, save_path=config.FIGURES_DIR / "tabpfn3_marginal_gain.png")

In [ ]:
best_n = int(sweep_df.loc[sweep_df["f1_macro"].idxmax(), "n_estimators"])
print("Best n_estimators by F1 macro:", best_n)
best_clf, tab_res = experiments.tabpfn_fit_best(X_train, X_test, y_train, y_test, best_n)
evaluate.print_classification_report(y_test, tab_res["y_pred"],
                                     f"TabPFN-3 (n_estimators={best_n}) - per-class report")
evaluate.plot_confusion_matrix(y_test, tab_res["y_pred"], f"TabPFN-3 (n_estimators={best_n})",
                               cmap="Oranges",
                               save_path=config.FIGURES_DIR / "tabpfn3_confusion_matrix.png")

### TabPFN-3 — permutation feature importance

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
perm = permutation_importance(best_clf, X_test, y_test, n_repeats=10,
                              scoring="f1_macro", random_state=config.RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns)
evaluate.plot_feature_importance(imp, "TabPFN-3 - Feature Importance (permutation)",
                                 color="darkorange", kind="bar",
                                 save_path=config.FIGURES_DIR / "tabpfn3_feature_importance.png")

## 2.3 · AutoGluon — 2x2 grid search

Varying `num_bag_folds` in {3, 8} x `num_stack_levels` in {1, 2}. Selection uses
AutoGluon's internal CV score (`score_val`); the test set is reported only for
reference. **Slow:** ~`time_limit` seconds per cell of the grid.

In [ ]:
train_data = models.make_autogluon_train_frame(X_train, y_train)
test_data  = models.make_autogluon_train_frame(X_test, y_test)
grid_df = experiments.autogluon_grid_search(train_data, test_data, y_test,
                                            folds=(3, 8), stacks=(1, 2), time_limit=300)
grid_df

In [ ]:
plots.plot_grid_heatmaps(grid_df, save_dir=config.FIGURES_DIR)

### Final evaluation of the best AutoGluon config

The CV-selected winner is `folds=8, stacks=2`; we fix it explicitly so this
always evaluates the configuration reported in the thesis. Its predictions are
saved to `results/autogluon_results.pkl`, which notebook **3** loads.

In [ ]:
best_pred, ag_res = experiments.autogluon_fit_best(train_data, test_data, y_test,
                                                  best_folds=8, best_stacks=2, time_limit=900)
evaluate.print_classification_report(y_test, ag_res["y_pred"], "AutoGluon (8/2) - per-class report")
evaluate.plot_confusion_matrix(y_test, ag_res["y_pred"], "AutoGluon (folds=8, stacks=2)",
                               cmap="Greens",
                               save_path=config.FIGURES_DIR / "autogluon_confusion_matrix.png")
fi = best_pred.feature_importance(test_data)["importance"]
evaluate.plot_feature_importance(fi, "AutoGluon - Feature Importance (permutation)",
                                 color="seagreen", kind="barh",
                                 save_path=config.FIGURES_DIR / "autogluon_feature_importance.png")